# Credit Card Fraud Detection — Exploratory Data Analysis

This notebook explores the **Kaggle Credit Card Fraud Detection** dataset, which contains 284,807 transactions made by European cardholders in September 2013. Of these, only 492 are fraudulent — representing just **0.172%** of all transactions.

Because the dataset is so highly imbalanced, standard accuracy is misleading. Our goal is to understand the data distributions, identify features that discriminate fraud from legitimate transactions, and prepare for modeling with appropriate sampling techniques.

In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_palette('husl')
%matplotlib inline

print('Libraries loaded successfully')

## Load Data

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
# Load data - user should place creditcard.csv in data/
DATA_PATH = '../data/creditcard.csv'
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    # Generate synthetic data for demonstration
    np.random.seed(42)
    n = 10000
    fraud_n = 17
    normal = pd.DataFrame(np.random.randn(n - fraud_n, 30), columns=[f'V{i}' for i in range(1,29)] + ['Time','Amount'])
    fraud = pd.DataFrame(np.random.randn(fraud_n, 30) * 2 + 3, columns=[f'V{i}' for i in range(1,29)] + ['Time','Amount'])
    normal['Class'] = 0
    fraud['Class'] = 1
    df = pd.concat([normal, fraud], ignore_index=True)
    print("Using synthetic demo data (place creditcard.csv in data/ for real analysis)")
print(df.shape)
df.head()

## Dataset Overview

In [ ]:
print('=== DataFrame Info ===')
df.info()
print('\n=== Descriptive Statistics ===')
df.describe()

## Class Distribution

In [ ]:
class_counts = df['Class'].value_counts()
fraud_rate = df['Class'].mean() * 100

print(f'Total transactions : {len(df):,}')
print(f'Legitimate (0)     : {class_counts[0]:,} ({100 - fraud_rate:.3f}%)')
print(f'Fraudulent (1)     : {class_counts[1]:,} ({fraud_rate:.3f}%)')
print(f'\nFraud rate: {fraud_rate:.4f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
sns.countplot(x='Class', data=df, ax=axes[0], palette=['steelblue', 'tomato'])
axes[0].set_title('Transaction Class Distribution', fontsize=13)
axes[0].set_xticklabels(['Legitimate', 'Fraud'])
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Pie chart
axes[1].pie(
    class_counts.values,
    labels=['Legitimate', 'Fraud'],
    autopct='%1.3f%%',
    colors=['steelblue', 'tomato'],
    startangle=90,
)
axes[1].set_title('Class Proportion', fontsize=13)

plt.tight_layout()
plt.show()

## Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Amount distribution by class
for cls, color, label in [(0, 'steelblue', 'Legitimate'), (1, 'tomato', 'Fraud')]:
    axes[0, 0].hist(
        df[df['Class'] == cls]['Amount'],
        bins=50, alpha=0.7, color=color, label=label, density=True
    )
axes[0, 0].set_title('Transaction Amount Distribution by Class', fontsize=12)
axes[0, 0].set_xlabel('Amount')
axes[0, 0].set_ylabel('Density')
axes[0, 0].legend()
axes[0, 0].set_xlim(0, df['Amount'].quantile(0.99))

# Log-scaled Amount
log_amount = df['Amount'].apply(lambda x: np.log1p(x))
for cls, color, label in [(0, 'steelblue', 'Legitimate'), (1, 'tomato', 'Fraud')]:
    axes[0, 1].hist(
        log_amount[df['Class'] == cls],
        bins=50, alpha=0.7, color=color, label=label, density=True
    )
axes[0, 1].set_title('Log(Amount+1) Distribution by Class', fontsize=12)
axes[0, 1].set_xlabel('log(Amount + 1)')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend()

# Time distribution by class
for cls, color, label in [(0, 'steelblue', 'Legitimate'), (1, 'tomato', 'Fraud')]:
    axes[1, 0].hist(
        df[df['Class'] == cls]['Time'],
        bins=48, alpha=0.7, color=color, label=label, density=True
    )
axes[1, 0].set_title('Time Distribution by Class', fontsize=12)
axes[1, 0].set_xlabel('Time (seconds from first transaction)')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()

# Box plot of Amount by class
df_plot = df[df['Amount'] < df['Amount'].quantile(0.99)].copy()
df_plot['Class_label'] = df_plot['Class'].map({0: 'Legitimate', 1: 'Fraud'})
sns.boxplot(x='Class_label', y='Amount', data=df_plot, ax=axes[1, 1],
            palette=['steelblue', 'tomato'])
axes[1, 1].set_title('Amount Box Plot by Class (< 99th pct)', fontsize=12)
axes[1, 1].set_xlabel('Class')
axes[1, 1].set_ylabel('Amount')

plt.suptitle('Feature Distributions: Amount and Time', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Correlation Analysis

In [ ]:
# Compute absolute correlation with target
corr_with_class = df.corr()['Class'].abs().drop('Class').sort_values(ascending=False)
top15_features = corr_with_class.head(15).index.tolist()

print('Top 15 features most correlated with Class (|r|):')
print(corr_with_class.head(15).to_string())

# Correlation heatmap of top features + Class
top_cols = top15_features + ['Class']
corr_matrix = df[top_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={'size': 8},
)
plt.title('Correlation Heatmap — Top 15 Features Most Correlated with Fraud', fontsize=13)
plt.tight_layout()
plt.show()

## Missing Values Check

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(4)

missing_summary = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print('Missing value summary:')
print(missing_summary[missing_summary['Missing Count'] > 0])

if missing.sum() == 0:
    print('\nNo missing values found in the dataset.')
else:
    print(f'\nTotal missing values: {missing.sum()}')

## Key Findings

- **Dataset is highly imbalanced** (~0.17% fraud) — accuracy alone is a meaningless metric; we need precision, recall, and AUC-ROC
- **PCA-transformed features (V1-V28)** show clear separation between classes; several features (e.g., V14, V12, V10, V4) have the strongest correlation with fraud
- **Amount and Time are raw features** — fraudulent transactions tend to involve smaller amounts and occur at specific times
- **No missing values** — the dataset is clean and ready for modeling
- **SMOTE will be applied** to the training set to address class imbalance before model training; it is never applied to the test set to avoid data leakage